# Pegasus Workflow Visualizer — Demo

This notebook demonstrates all major features of the `workflow-visualizer` package.
Each section is self-contained — run only the cells relevant to your use case.

## Source Modes

| Mode       | Badge    | Description                                              |
|------------|----------|----------------------------------------------------------|
| **Static** | `STATIC` | DAG structure only — no live event feed                  |
| **Live**   | `LIVE`   | Polling a local `workflow-events.jsonl` file             |
| **SSH**    | `SSH`    | Fetching events from a remote host over SSH              |

## Prerequisites

```bash
# Install the package (from the repo root)
uv pip install -e .

# Or with dev dependencies
uv pip install -e '.[dev]'
```

In [ ]:
from workflow_visualizer import WorkflowVisualizerWidget

---
## 1. Static DAG from `workflow.yml`

Render a Pegasus workflow DAG from its YAML definition — no live events.
The toolbar shows the **STATIC** badge (gray).

Point `WORKFLOW_YML` at any Pegasus `workflow.yml` file on your system.

In [ ]:
# -- Edit this path to point at your workflow.yml --
WORKFLOW_YML = "path/to/your/workflow.yml"

w_static = WorkflowVisualizerWidget(workflow_path=WORKFLOW_YML)
w_static

### 1b. Static DAG with file nodes visible

Enable `show_files=True` to render data file nodes alongside compute jobs.
File nodes appear as smaller rectangles showing data flow through the DAG.

In [ ]:
w_static_files = WorkflowVisualizerWidget(
    workflow_path=WORKFLOW_YML,
    show_files=True,
)
w_static_files

### 1c. Toggle file nodes on an existing widget

Use `.show()` to toggle or explicitly set file node visibility on a widget
that is already rendered.

In [ ]:
# Toggle show_files (flips current value)
w_static.show()

# Or explicitly enable/disable
# w_static.show(show_files=True)
# w_static.show(show_files=False)

### 1d. Text summary of the DAG

Print a quick overview of the workflow structure: job count, edges, inputs, and outputs.

In [ ]:
w_static.summary()

---
## 2. Live DAG with local event feed

Feed a local `workflow-events.jsonl` file to see nodes animate through states.
The toolbar shows the **LIVE** badge (green, pulsing) while polling.

This mode requires:
1. A `workflow.yml` file (for DAG structure)
2. A `workflow-events.jsonl` file (produced by `workflow-monitor`)

The widget polls the JSONL file every `poll_interval` seconds and updates
node colors based on UML state machine conventions.

In [ ]:
# -- Edit these paths --
WORKFLOW_YML = "path/to/your/workflow.yml"
EVENTS_JSONL = "path/to/your/workflow-events.jsonl"

w_live = WorkflowVisualizerWidget(
    workflow_path=WORKFLOW_YML,
    jsonl_path=EVENTS_JSONL,
    poll_interval=2.0,
)
w_live

### 2b. Live DAG with file nodes

Same as above but with data file nodes visible, showing stage-in/stage-out progress.

In [ ]:
w_live_files = WorkflowVisualizerWidget(
    workflow_path=WORKFLOW_YML,
    jsonl_path=EVENTS_JSONL,
    poll_interval=2.0,
    show_files=True,
)
w_live_files

### 2c. Auto-refresh with `watch()`

Use `.watch()` for a pseudo-live display that redraws in-place using
`clear_output`. This works in environments where the anywidget comm channel
is unavailable (e.g., ACCESS classic Notebook).

Press **Interrupt Kernel** to stop watching.

In [ ]:
# Auto-refresh until workflow completes (Interrupt Kernel to stop)
w_live.watch()

# Or with custom interval and file nodes
# w_live.watch(interval=5, show_files=True)

### 2d. Manual polling control

Start and stop background polling explicitly.

In [ ]:
# Start background polling
w_live.start_polling()

# ... observe the widget updating ...

# Stop when done
# w_live.stop_polling()

---
## 3. Live DAG without `workflow.yml` (event-driven graph)

If no `workflow.yml` is available, the widget can build a graph purely from
the JSONL event stream. Nodes are discovered as events arrive.

This is useful when monitoring a workflow where only the event file is accessible.

In [ ]:
# -- Edit this path --
EVENTS_JSONL = "path/to/your/workflow-events.jsonl"

w_events_only = WorkflowVisualizerWidget(
    jsonl_path=EVENTS_JSONL,
    poll_interval=2.0,
)
w_events_only

---
## 4. Remote SSH workflow

Connect to a remote Pegasus submit node over SSH to monitor a running workflow.
The toolbar shows the **SSH** badge (purple, pulsing) with the remote host on hover.

### SSH prerequisites

Ensure you can `ssh <host>` to the submit node. For jump-host scenarios
(e.g., FABRIC testbed), configure your SSH config file accordingly.

### `remote_spec` format

```
user@host:/path/to/workflow-events.jsonl
```

Or with an SSH config alias:

```
my-host-alias:/path/to/workflow-events.jsonl
```

### 4a. SSH with config alias

If your `~/.ssh/config` (or a custom config file) defines a host alias,
use it as the host portion of `remote_spec`.

In [ ]:
# -- Edit these values --
WORKFLOW_YML = "path/to/workflow.yml"  # local or remote path to workflow definition
REMOTE_SPEC = "my-ssh-alias:/path/to/workflow-events.jsonl"
SSH_CONFIG = "~/.ssh/config"           # path to SSH config (optional)
SSH_IDENTITY = "~/.ssh/id_rsa"         # path to SSH key (optional)

w_ssh = WorkflowVisualizerWidget(
    workflow_path=WORKFLOW_YML,
    remote_spec=REMOTE_SPEC,
    ssh_config=SSH_CONFIG,
    ssh_identity=SSH_IDENTITY,
    poll_interval=5.0,
)
w_ssh

### 4b. SSH with explicit user@host

Connect directly using `user@hostname` without an SSH config alias.

In [ ]:
# -- Edit these values --
REMOTE_SPEC = "ubuntu@submit-node.example.com:/home/ubuntu/my-workflow/workflow-events.jsonl"
SSH_IDENTITY = "~/.ssh/id_rsa"

w_ssh_direct = WorkflowVisualizerWidget(
    remote_spec=REMOTE_SPEC,
    ssh_identity=SSH_IDENTITY,
    poll_interval=5.0,
)
w_ssh_direct

### 4c. SSH with IPv6 address

For environments like FABRIC JupyterHub where nodes are on the same slice,
use an IPv6 address wrapped in brackets.

In [ ]:
# -- Edit these values --
# REMOTE_SPEC = "ubuntu@[2001:db8::1]:/home/ubuntu/my-workflow/workflow-events.jsonl"
# SSH_CONFIG = "~/.ssh/fabric-ssh-config"
# SSH_IDENTITY = "~/.ssh/my-sliver-key"
#
# w_ssh_ipv6 = WorkflowVisualizerWidget(
#     remote_spec=REMOTE_SPEC,
#     ssh_config=SSH_CONFIG,
#     ssh_identity=SSH_IDENTITY,
#     poll_interval=5.0,
# )
# w_ssh_ipv6

---
## 5. FABRIC Testbed Example

FABRIC nodes are accessed through a bastion host. This section shows the
recommended SSH configuration and widget setup.

### SSH config for FABRIC

Create or update your SSH config (e.g., `~/.ssh/fabric-ssh-config`):

```
UserKnownHostsFile /dev/null
StrictHostKeyChecking no
ServerAliveInterval 120

Host bastion.fabric-testbed.net
    User <your-fabric-username>
    IdentityFile <path-to-bastion-key>
    ForwardAgent yes

Host pegasus-submit
    User ubuntu
    HostName <submit-node-ip>
    ProxyJump bastion.fabric-testbed.net
    IdentityFile <path-to-sliver-key>
```

With this config, `ssh pegasus-submit` tunnels through the bastion automatically.

In [ ]:
# -- Edit these values for your FABRIC slice --
# Ensure workflow-monitor is serving on the submit node first:
#   ssh pegasus-submit "uv run workflow-monitor --serve /home/ubuntu/my-workflow"

# WORKFLOW_YML = "/home/ubuntu/my-workflow/workflow.yml"  # path on the remote host
# REMOTE_SPEC = "pegasus-submit:/home/ubuntu/my-workflow/ubuntu/pegasus/wf/run0001/workflow-events.jsonl"
# SSH_CONFIG = "~/.ssh/fabric-ssh-config"
# SSH_IDENTITY = "~/.ssh/my-sliver-key"
#
# w_fabric = WorkflowVisualizerWidget(
#     workflow_path=WORKFLOW_YML,
#     remote_spec=REMOTE_SPEC,
#     ssh_config=SSH_CONFIG,
#     ssh_identity=SSH_IDENTITY,
#     poll_interval=5.0,
# )
# w_fabric

---
## 6. Workflow Lifecycle Controls

The widget wraps Pegasus CLI commands for planning, running, stopping, and
resuming workflows. These require Pegasus to be installed and accessible.

Provide a `submit_dir` to enable lifecycle controls.

In [ ]:
from workflow_visualizer.controls import WorkflowControls

# -- Edit this path --
SUBMIT_DIR = "path/to/your/submit-dir"

ctl = WorkflowControls(submit_dir=SUBMIT_DIR)

### 6a. Plan a workflow

Run `pegasus-plan` on a DAX/workflow file.

In [ ]:
# DAX_FILE = "path/to/your/workflow.yml"
# result = ctl.plan(DAX_FILE)
# print(result)

### 6b. Run / Stop / Resume

In [ ]:
# Start the workflow
# result = ctl.run()
# print("Run:", result)

# Stop (remove) the workflow
# result = ctl.stop()
# print("Stop:", result)

# Resume a stopped workflow
# result = ctl.resume()
# print("Resume:", result)

### 6c. Workflow Monitor controls

Start and stop the `workflow-monitor` server that produces `workflow-events.jsonl`.

In [ ]:
# Start workflow-monitor
# result = ctl.monitor_start()
# print("Monitor start:", result)

# Stop workflow-monitor
# result = ctl.monitor_stop()
# print("Monitor stop:", result)

---
## 7. Using the Parser and State Modules Directly

The underlying modules can be used independently for custom analysis.

### 7a. Parse a workflow YAML

Use `WorkflowGraph.from_yaml()` to parse a Pegasus workflow definition into
a graph structure with nodes, edges, and metadata.

In [ ]:
from workflow_visualizer.parser import WorkflowGraph

# -- Edit this path --
WORKFLOW_YML = "path/to/your/workflow.yml"

graph = WorkflowGraph.from_yaml(WORKFLOW_YML)

print(f"Workflow: {graph.metadata['name']}")
print(f"Pegasus:  {graph.metadata['pegasus_version']}")
print(f"Nodes:    {len(graph.nodes)}")
print(f"Edges:    {len(graph.edges)}")
print()

# Inspect nodes
for node in graph.nodes:
    print(f"  {node['id']:20s}  label={node['nodeLabel']}  "
          f"inputs={node['inputs']}  outputs={node['outputs']}")

print()
# Inspect edges
for edge in graph.edges:
    print(f"  {edge['source']} -> {edge['target']}")

### 7b. Serialize graph to dict

The `.to_dict()` method produces a JSON-compatible dictionary suitable
for the widget traitlet or external tools.

In [ ]:
import json

graph_dict = graph.to_dict()
print(json.dumps(graph_dict, indent=2)[:2000])  # first 2000 chars

### 7c. State colors and formatting utilities

In [ ]:
from workflow_visualizer.state import (
    STATE_COLORS,
    STATE_MAP,
    JOB_TYPE_LABEL,
    display_state,
    state_color,
    fmt_duration,
    fmt_timestamp,
    fmt_memory,
    fmt_memory_mb,
    fmt_bytes,
    fmt_percent,
)

# UML state color palette
print("State Colors (UML state machine convention):")
for st, colors in STATE_COLORS.items():
    print(f"  {st:14s}  fill={colors['fill']}  stroke={colors['stroke']}")

print()

# Raw state -> display state mapping
print("State Mapping (Pegasus raw -> display):")
for raw, disp in STATE_MAP.items():
    print(f"  {raw:25s} -> {disp}")

print()

# Job type labels
print("Job Type Labels:")
for jtype, label in JOB_TYPE_LABEL.items():
    print(f"  {jtype:16s} -> {label}")

In [ ]:
# Formatting utilities
print("Duration formatting:")
for secs in [0, 45, 125, 3661, 7200]:
    print(f"  {secs:6d}s -> {fmt_duration(secs)}")

print()
print("Memory formatting (KB):")
for kb in [512, 2048, 1048576, 5242880]:
    print(f"  {kb:>10d} KB -> {fmt_memory(kb)}")

print()
print("Memory formatting (MB):")
for mb in [256, 1024, 4096]:
    print(f"  {mb:>6d} MB -> {fmt_memory_mb(mb)}")

print()
print("Bytes formatting:")
for b in [100, 1500, 1500000, 1500000000]:
    print(f"  {b:>12d} B -> {fmt_bytes(b)}")

print()
print("Percentage formatting:")
for ratio in [0.0, 0.25, 0.756, 1.0]:
    print(f"  {ratio:.3f} -> {fmt_percent(ratio)}")

### 7d. Event consumer (standalone)

Use the `EventConsumer` directly to read and process a JSONL event file.

In [ ]:
from workflow_visualizer.events import EventConsumer

# -- Edit this path --
EVENTS_JSONL = "path/to/your/workflow-events.jsonl"

consumer = EventConsumer(EVENTS_JSONL)

# Read all available events
has_events = consumer.poll()
print(f"Events found: {has_events}")
print(f"Workflow state: {consumer.workflow_state}")
print(f"Complete: {consumer.is_complete}")
print(f"Jobs tracked: {len(consumer.job_states)}")
print(f"Event log entries: {len(consumer.event_log)}")

print()
# Show the last 5 events
print("Recent events:")
for evt in consumer.event_log[:5]:
    print(f"  {evt.get('event_name', '?'):25s}  "
          f"state={evt.get('event_state', '?'):15s}  "
          f"time={evt.get('event_start_time', '')}")

---
## 8. No workflow file — placeholder UI

When no `workflow_path` is provided, the widget displays a placeholder
with a file chooser. Enter a path to `workflow.yml` interactively.

In [ ]:
w_placeholder = WorkflowVisualizerWidget()
w_placeholder

---
## 9. Full workflow: plan, monitor, and visualize

End-to-end example combining lifecycle controls with live visualization.
This requires Pegasus and HTCondor to be installed and running locally.

In [ ]:
# -- Edit these paths --
# WORKFLOW_YML = "path/to/your/workflow.yml"  # Pegasus workflow definition
# SUBMIT_DIR = "path/to/your/submit-dir"      # will be created by pegasus-plan
# EVENTS_JSONL = "path/to/your/submit-dir/workflow-events.jsonl"
#
# # Step 1: Plan the workflow
# from workflow_visualizer.controls import WorkflowControls
# ctl = WorkflowControls(submit_dir=SUBMIT_DIR)
# plan_result = ctl.plan(WORKFLOW_YML)
# print("Plan result:", plan_result["status"])
#
# # Step 2: Start workflow-monitor
# monitor_result = ctl.monitor_start()
# print("Monitor:", monitor_result["status"])
#
# # Step 3: Run the workflow
# run_result = ctl.run()
# print("Run:", run_result["status"])
#
# # Step 4: Visualize with live updates
# w_full = WorkflowVisualizerWidget(
#     workflow_path=WORKFLOW_YML,
#     jsonl_path=EVENTS_JSONL,
#     submit_dir=SUBMIT_DIR,
#     poll_interval=2.0,
# )
# w_full.watch()  # auto-refresh until complete

---
## 10. Cleanup

Always close widgets when done to stop polling threads and clean up resources.

In [ ]:
# Close all widgets created in this notebook
for w in [w for w in dir() if w.startswith('w_') and hasattr(eval(w), 'close')]:
    eval(w).close()
    print(f"Closed {w}")